In [5]:
import pandas as pd 

orders = pd.read_excel("Sample - Superstore Sales.xls",sheet_name='Orders')
returns = pd.read_excel("Sample - Superstore Sales.xls",sheet_name='Returns')
users = pd.read_excel("Sample - Superstore Sales.xls",sheet_name='Users')
print(pd.__version__)
print(orders.head())
print('-' * 100)
print(orders.tail())
print('-' * 100)
print(orders.shape)
print('-' * 100)
print(orders.dtypes)
print('-' * 100)
print(orders.memory_usage(deep=True).sort_values(ascending=False)) #Результат показує обсяг пам'яті, який забирає кожна колонка. Найбільше займає колонка зі стрічками, тому що, під них немає сталої резервації пам'яті
print('-' * 100)
print(orders.describe())
print('-' * 100)
print(orders.describe(include="object"))
print('-' * 100)
print(orders['Product Category'].nunique())
print(orders['Product Sub-Category'].nunique())
print(orders['Region'].nunique())
orders["Ship Mode"].value_counts(normalize=True) * 100
print('=' * 100)
orders[['Order ID', 'Sales', 'Profit']]
print('=' * 100)
orders.loc[:, ['Order ID', 'Sales', 'Profit']]
print(orders.iloc[100:111])
print("=" *100)
print(orders.loc[:19, "Profit":"Product Category"])
orders_indexed = orders.set_index("Order ID")
orders_indexed.loc[3]


print(orders.isna().sum())
 
import numpy as np
orders_na = orders.copy()
print("Пропусків до:", orders_na["Product Base Margin"].isna().sum())
rng = np.random.default_rng(42)
mask = rng.random(len(orders_na)) < 0.05
orders_na.loc[mask, "Product Base Margin"] = np.nan
print("Пропусків після:", orders_na["Product Base Margin"].isna().sum())
 

mode_val = orders_na["Product Base Margin"].mode()[0]
orders_na["Product Base Margin"] = orders_na["Product Base Margin"].fillna(mode_val)
print("Пропусків після заповнення:", orders_na["Product Base Margin"].isna().sum())
 

disc = orders["Discount"].copy()
mask_d = rng.random(len(disc)) < 0.05
disc_na = disc.mask(mask_d)
fill_zero = disc_na.fillna(0)
fill_mean = disc_na.fillna(disc_na.mean())
fill_prev = disc_na.ffill()
compare = pd.DataFrame({
    "original": disc,
    "zero": fill_zero,
    "mean": fill_mean,
    "ffill": fill_prev,
})
print(compare.describe())
# Середня абсолютна помилка щодо оригіналу
print((compare[["zero", "mean", "ffill"]].sub(compare["original"], axis=0)).abs().mean())
 

orders_dropna = orders_na.copy()
orders_dropna.loc[mask, "Product Base Margin"] = np.nan  
dropped = orders_dropna.dropna(subset=["Product Base Margin"])
print("Рядків до:", len(orders_dropna), "після:", len(dropped))

order_id_before = orders["Order ID"].copy()
discount_before = orders["Discount"].copy()
print("Order ID до:", order_id_before.memory_usage(deep=True))
print("Order ID після (str):", order_id_before.astype(str).memory_usage(deep=True))
print("Discount до:", discount_before.memory_usage(deep=True))
print("Discount після (float32):", discount_before.astype("float32").memory_usage(deep=True))

names = orders["Customer Name"].str.split(" ", n=1, expand=True)
orders["First Name"] = names[0]
orders["Last Name"] = names[1]
 
orders["Product Name Length"] = orders["Product Name"].str.len()
print("Назв із 'Chair':", orders["Product Name"].str.contains("chair", case=False, na=False).sum())
 
orders["Province"] = orders["Province"].str.strip()
orders["Product Sub-Category"] = orders["Product Sub-Category"].str.replace("-", " ", regex=False)
print(orders[["First Name", "Last Name", "Product Name Length", "Province", "Product Sub-Category"]].head())

orders["Order Date"] = pd.to_datetime(orders["Order Date"])
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"])
 
orders["Order Year"] = orders["Order Date"].dt.year
orders["Order Month"] = orders["Order Date"].dt.month
orders["Order Weekday"] = orders["Order Date"].dt.day_name()
orders["Order Quarter"] = orders["Order Date"].dt.quarter
 
orders["Ship Delay"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
print(orders["Ship Delay"].describe())
 
orders_2012 = orders[orders["Order Year"] == 2012]
print("Замовлень 2012:", len(orders_2012))
 
monthly_sales = orders.set_index("Order Date")["Sales"].resample("MS").sum()
print(monthly_sales.head())
 
print(orders.groupby("Order Weekday")["Ship Delay"].mean())
 

orders["Profit Margin"] = orders["Profit"] / orders["Sales"]
 
orders = orders.assign(
    Sales_Per_Unit=orders["Sales"] / orders["Order Quantity"],
    Is_Profitable=orders["Profit"] > 0,
)
 

orders["Discount Level"] = np.select(
    [orders["Discount"] == 0, orders["Discount"] < 0.2],
    ["No", "Low"],
    default="High",
)

def discount_level(d):
    if d == 0:
        return "No"
    elif d < 0.2:
        return "Low"
    return "High"
 
level_apply = orders["Discount"].apply(discount_level)
print("Підходи збігаються:", (orders["Discount Level"] == level_apply).all())
print(orders["Discount Level"].value_counts())
 

region_codes = {
    "West": "W", "Atlantic": "ATL", "Ontario": "ON", "Quebec": "QC",
    "Prairie": "PR", "Nunavut": "NU", "Northwest Territories": "NT", "Yukon": "YT",
}
orders["Region Code"] = orders["Region"].map(region_codes).fillna(orders["Region"])
 

num_cols = orders.select_dtypes(include="number").columns
orders[num_cols] = orders[num_cols].round(2)

orders["Ship Speed"] = np.select(
    [orders["Ship Delay"] <= 2, orders["Ship Delay"].between(3, 5)],
    ["швидке", "звичайне"],
    default="повільне",
)
print(orders["Ship Speed"].value_counts())

print(orders.sort_values("Sales", ascending=False).head())
print(orders.sort_values(["Region", "Profit"], ascending=[True, False]).head())
print(orders.nlargest(10, "Profit")[["Order ID", "Profit"]])
print(orders.nsmallest(10, "Profit")[["Order ID", "Profit"]])
 
orders["Sales Rank in Region"] = (
    orders.groupby("Region")["Sales"].rank(ascending=False, method="min")
)

print(orders.groupby("Product Category")[["Sales", "Profit"]].sum())
 
region_stats = orders.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Avg_Discount=("Discount", "mean"),
    Unique_Orders=("Order ID", "nunique"),
    Max_Profit=("Profit", "max"),
)
print(region_stats)
 
cat_sub_profit = orders.groupby(["Product Category", "Product Sub-Category"])["Profit"].sum()  # завдання 46
print(cat_sub_profit)
 
orders["Category Avg Sales"] = orders.groupby("Product Category")["Sales"].transform("mean")
orders["Sales Deviation"] = orders["Sales"] - orders["Category Avg Sales"]
 
sub_profit = orders.groupby("Product Sub-Category")["Profit"].sum()
profitable_subs = sub_profit[sub_profit > 0].index
orders_profitable = orders[orders["Product Sub-Category"].isin(profitable_subs)]
print(len(orders), "->", len(orders_profitable))
 
segment_stats = orders.groupby("Customer Segment").agg(
    Unique_Orders=("Order ID", "nunique"),
    Total_Sales=("Sales", "sum"),
    Avg_Sales=("Sales", "mean"),
    Total_Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean"),
)
print(segment_stats)
 
reg_sub = orders.groupby(["Region", "Product Sub-Category"], as_index=False)["Profit"].sum()
best_sub_per_region = reg_sub.loc[reg_sub.groupby("Region")["Profit"].idxmax()]
print(best_sub_per_region)

pivot = orders.pivot_table(
    values="Sales", index="Region", columns="Product Category",
    aggfunc="sum", margins=True, margins_name="Total",
)  
print(pivot)
 
print(pd.crosstab(orders["Customer Segment"], orders["Ship Mode"]))

pivot_long = (
    pivot.drop(index="Total", columns="Total")
    .reset_index()
    .melt(id_vars="Region", var_name="Product Category", value_name="Sales")
)
print(pivot_long.head())

wide = cat_sub_profit.unstack(level="Product Sub-Category")
print(wide)
back = wide.stack()
print(back.head())

try:
    orders.pivot(index="Region", columns="Product Category", values="Sales")
except ValueError as e:
    print("pivot() помилка:", e)
 

merged = orders.merge(returns, on="Order ID", how="left")
merged["Status"] = merged["Status"].fillna("No")
print(len(orders), len(merged))  
 
merged = merged.merge(users[["Region", "Manager"]], on="Region", how="left")
print(merged[["Order ID", "Region", "Status", "Manager"]].head())
 
print(merged.groupby("Manager")["Sales"].sum())
 

left_df = pd.DataFrame({"key": [1, 2, 3], "left_val": ["a", "b", "c"]})
right_df = pd.DataFrame({"key": [2, 3, 4], "right_val": ["x", "y", "z"]})
for how in ["inner", "left", "right", "outer"]:
    print(f"--- {how} ---")
    print(left_df.merge(right_df, on="key", how=how))

part1 = orders[orders["Order Year"] <= 2010]
part2 = orders[orders["Order Year"] > 2010]
restored = pd.concat([part1, part2])
print(len(restored) == len(orders), len(restored), len(orders))

print("Повних дублікатів:", orders.duplicated().sum())
orders_unique = orders.drop_duplicates()
 
first_per_customer = orders.drop_duplicates(subset="Customer Name", keep="first")
print("Унікальних клієнтів:", len(first_per_customer))
 
orders["Sales Bin"] = pd.cut(orders["Sales"], bins=4, labels=["Low", "Medium", "High", "Very High"])
print(orders["Sales Bin"].value_counts().sort_index())
 
orders["Profit Quartile"] = pd.qcut(orders["Profit"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(orders.groupby("Profit Quartile", observed=True)["Sales"].mean())
 
before = orders[["Product Category", "Region"]].memory_usage(deep=True).sum()
cat_df = orders[["Product Category", "Region"]].astype("category")
after = cat_df.memory_usage(deep=True).sum()
print("Пам'ять до:", before, "після:", after)

sorted_orders = orders.sort_values("Order Date")
sorted_orders["Cumulative Sales"] = sorted_orders["Sales"].cumsum()
print(sorted_orders[["Order Date", "Sales", "Cumulative Sales"]].head())
 
rolling_mean = monthly_sales.rolling(window=3).mean()
expanding_mean = monthly_sales.expanding().mean()

print(pd.DataFrame({"sales": monthly_sales, "rolling_3": rolling_mean, "expanding": expanding_mean}).head(10))
 
change = pd.DataFrame({
    "pct_change": monthly_sales.pct_change() * 100,
    "abs_change": monthly_sales.diff(),
})
print(change.head())
 

g3 = orders.groupby(["Region", "Product Category", "Product Sub-Category"])["Sales"].sum()
print(g3.head())
print(g3.loc["West"])                              
swapped = g3.swaplevel(0, 1).sort_index()       
print(swapped.head())
flat = swapped.reset_index()                      
print(flat.head())




2.3.3
   Row ID  Order ID Order Date Order Priority  Order Quantity       Sales  \
0       1         3 2010-10-13            Low               6    261.5400   
1      49       293 2012-10-01           High              49  10123.0200   
2      50       293 2012-10-01           High              27    244.5700   
3      80       483 2011-07-10           High              30   4965.7595   
4      85       515 2010-08-28  Not Specified              19    394.2700   

   Discount       Ship Mode     Profit  Unit Price  ...       Customer Name  \
0      0.04     Regular Air  -213.2500       38.94  ...  Muhammed MacIntyre   
1      0.07  Delivery Truck   457.8100      208.16  ...        Barry French   
2      0.01     Regular Air    46.7075        8.69  ...        Barry French   
3      0.08     Regular Air  1198.9710      195.99  ...       Clay Rozendal   
4      0.08     Regular Air    30.9400       21.78  ...      Carlos Soltero   

  Province   Region Customer Segment Product Category  \